In [ ]:
import napari
import os
import numpy as np
import matplotlib.pyplot as plt

while os.path.basename(os.getcwd()) != 'photostim_deve':
    os.chdir('..')

from cv2 import VideoCapture

In [ ]:
subject_data_vid_path = 'data_vid/jm064'

In [ ]:
# now get all .avi filenames and sort them alphabetically
vid_filenames = [f for f in os.listdir(subject_data_vid_path) if f.endswith('.avi') and not f.startswith('.')]
vid_filenames.sort()
print(f'Found {len(vid_filenames)} .avi files in {subject_data_vid_path}:')
for vid_filename in vid_filenames:
    print(vid_filename)

In [ ]:
# now load the first frame of each file
frames = []
for vid_filename in vid_filenames:
    vid_path = os.path.join(subject_data_vid_path, vid_filename)
    # load the first frame of the video
    cap = VideoCapture(vid_path)
    ret, frame = cap.read()
    if ret:
        frames.append(frame)
    cap.release()

In [ ]:
frames = np.array(frames)


In [ ]:
for i in range(len(frames)):
    plt.imshow(frames[i])
    plt.show()

In [ ]:
LANDMARKS = ["headplate_a", "headplate_p", "snout", "ear_a", "ear_p", "eye", "mouth"]

stack = np.stack(frames)          # (6, H, W, 3) from your cv2 frames
viewer = napari.Viewer()
viewer.add_image(stack, rgb=True, name="frames")

points = viewer.add_points(
    ndim=3,          # (frame, y, x)
    name="landmarks",
    size=8,
    face_color="cyan",
)

napari.run()

In [ ]:
import pandas as pd

n_frames = stack.shape[0]
n_landmarks = len(LANDMARKS)
data = points.data  # (N, 3): frame, y, x, in click order

assert len(data) == n_frames * n_landmarks, (
    f"expected {n_frames * n_landmarks} points, got {len(data)}"
)

rows = {frame: {"frame": frame} for frame in range(n_frames)}
for i, (frame, y, x) in enumerate(data):
    landmark = LANDMARKS[i // n_frames]   # which block of n_frames clicks this belongs to
    rows[int(frame)][f"{landmark}_x"] = x
    rows[int(frame)][f"{landmark}_y"] = y

df = pd.DataFrame(rows.values()).sort_values("frame").reset_index(drop=True)

In [ ]:
# if it doesnt exist make a directory in the subject to save the csv and plots
output_dir = os.path.join(subject_data_vid_path, "landmarks_outputs")
if not os.path.exists(output_dir):
    os.makedirs(output_dir)

In [ ]:
df.to_csv(os.path.join(subject_data_vid_path, "landmarks.csv"), index=False)

In [ ]:
# now visualise it based on the data in the table:
for i in range(len(frames)):
    plt.imshow(frames[i])
    plt.scatter(df.loc[i, [f"{lm}_x" for lm in LANDMARKS]], df.loc[i, [f"{lm}_y" for lm in LANDMARKS]], c='C0', s=10)
    # add text
    for lm in LANDMARKS:
        plt.text(df.loc[i, f"{lm}_x"]+10, df.loc[i, f"{lm}_y"]+10, lm, color='C0', fontsize=8)
    plt.title(f"P{i+8}")
    plt.axis('off')
    plt.savefig(os.path.join(output_dir, f"P{i+8}.png"), dpi=300, bbox_inches='tight')
    plt.show()

In [ ]:
from itertools import combinations
import matplotlib.pyplot as plt
import numpy as np

pairs = list(combinations(LANDMARKS, 2))

dist_df = pd.DataFrame({"frame": df["frame"]})
for a, b in pairs:
    dx = df[f"{a}_x"] - df[f"{b}_x"]
    dy = df[f"{a}_y"] - df[f"{b}_y"]
    dist_df[f"{a}-{b}"] = np.sqrt(dx**2 + dy**2)

pair_cols = dist_df.columns[1:]

colors = plt.rcParams['axes.prop_cycle'].by_key()['color']  # 10 default colors
markers = ['o', 's', '^', 'D', 'v', 'P', 'X', '*', 'h', '<', '>']

def plot_lines(ax, x, plot_df):
    for i, col in enumerate(pair_cols):
        color = colors[i % len(colors)]
        marker = markers[i // len(colors)]
        ax.plot(x, plot_df[col], color=color, marker=marker, label=col)

# 1. absolute distance
fig, ax = plt.subplots(figsize=(4, 6))
plot_lines(ax, dist_df["frame"], dist_df)
ax.set_xlabel("frame"); ax.set_ylabel("distance (px)")
ax.set_title("Pairwise landmark distances across frames")
# ax.legend(bbox_to_anchor=(1.05, 1), loc="upper left", fontsize=8)
plt.tight_layout()
plt.savefig(os.path.join(output_dir, "pairwise_distances_absolute.png"), dpi=300, bbox_inches='tight')
plt.show()

# 2. relative (baseline-subtracted)
rel_df = dist_df.copy()
rel_df[pair_cols] = dist_df[pair_cols] - dist_df[pair_cols].iloc[0]

fig, ax = plt.subplots(figsize=(4, 6))
plot_lines(ax, rel_df["frame"], rel_df)
ax.axhline(0, color="gray", lw=0.8, ls="--")
ax.set_xlabel("frame"); ax.set_ylabel("distance change from frame 0 (px)")
ax.set_title("Relative pairwise distance (baseline-subtracted)")
# ax.legend(bbox_to_anchor=(1.05, 1), loc="upper left", fontsize=8)
plt.tight_layout()
plt.savefig(os.path.join(output_dir, "pairwise_distances_relative.png"), dpi=300, bbox_inches='tight')
plt.show()

# 3. percentage of frame 0
pct_df = dist_df.copy()
pct_df[pair_cols] = dist_df[pair_cols] / dist_df[pair_cols].iloc[0] * 100

fig, ax = plt.subplots(figsize=(4, 6))
plot_lines(ax, pct_df["frame"], pct_df)
ax.axhline(100, color="gray", lw=0.8, ls="--")
ax.set_xlabel("frame"); ax.set_ylabel("% of frame 0 distance")
ax.set_title("Relative pairwise distance (% of frame 0)")
# ax.legend(bbox_to_anchor=(1.05, 1), loc="upper left", fontsize=8)
plt.tight_layout()
plt.savefig(os.path.join(output_dir, "pairwise_distances_percentage.png"), dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# --- calibrate to real-world units using headplate_a-headplate_p as a 5 mm reference ---
REF_PAIR = "headplate_a-headplate_p"
REF_MM = 5.0

scale_mm_per_px = REF_MM / dist_df[REF_PAIR]   # one scale factor per frame, corrects for
                                                # camera distance/angle changes across sessions

mm_df = dist_df.copy()
mm_df[pair_cols] = dist_df[pair_cols].multiply(scale_mm_per_px, axis=0)

# 4. absolute distance, calibrated (mm)
fig, ax = plt.subplots(figsize=(3, 3))
plot_lines(ax, mm_df["frame"], mm_df)
ax.set_xlabel("frame"); ax.set_ylabel("distance (mm)")
ax.set_title("Pairwise landmark distances across frames (calibrated)")
ax.legend(bbox_to_anchor=(1.05, 1), loc="upper left", fontsize=8)
plt.tight_layout()
plt.savefig(os.path.join(output_dir, "pairwise_distances_legen.png"), dpi=300, bbox_inches='tight')
plt.show()

# 4. absolute distance, calibrated (mm)
fig, ax = plt.subplots(figsize=(4, 6))
plot_lines(ax, mm_df["frame"], mm_df)
ax.set_xlabel("frame"); ax.set_ylabel("distance (mm)")
ax.set_title("Pairwise landmark distances across frames (calibrated)")
# ax.legend(bbox_to_anchor=(1.05, 1), loc="upper left", fontsize=8)
plt.tight_layout()
plt.savefig(os.path.join(output_dir, "pairwise_distances_absolute_calibrated.png"), dpi=300, bbox_inches='tight')
plt.show()

# 5. relative (baseline-subtracted), calibrated (mm)
rel_mm_df = mm_df.copy()
rel_mm_df[pair_cols] = mm_df[pair_cols] - mm_df[pair_cols].iloc[0]

fig, ax = plt.subplots(figsize=(4, 6))
plot_lines(ax, rel_mm_df["frame"], rel_mm_df)
ax.axhline(0, color="gray", lw=0.8, ls="--")
ax.set_xlabel("frame"); ax.set_ylabel("distance change from frame 0 (mm)")
ax.set_title("Relative pairwise distance, calibrated (baseline-subtracted)")
plt.tight_layout()
plt.savefig(os.path.join(output_dir, "pairwise_distances_relative_calibrated.png"), dpi=300, bbox_inches='tight')
plt.show()

# 6. percentage of frame 0, calibrated
pct_mm_df = mm_df.copy()
pct_mm_df[pair_cols] = mm_df[pair_cols] / mm_df[pair_cols].iloc[0] * 100

fig, ax = plt.subplots(figsize=(4, 6))
plot_lines(ax, pct_mm_df["frame"], pct_mm_df)
ax.axhline(100, color="gray", lw=0.8, ls="--")
ax.set_xlabel("frame"); ax.set_ylabel("% of frame 0 distance (calibrated)")
ax.set_title("Relative pairwise distance, calibrated (% of frame 0)")
plt.tight_layout()
plt.savefig(os.path.join(output_dir, "pairwise_distances_percentage_calibrated.png"), dpi=300, bbox_inches='tight')
plt.show()

: 